# Scenario 6: Structured Data Extraction

## 1. In One Minute

**What it is.** A pipeline extracts fields from unstructured documents (invoices, contracts, forms), validates them against JSON schemas, handles edge cases, and feeds downstream systems with high accuracy.

**Domains hit:** D4 Prompt Engineering and Structured Output, D5 Context Management and Reliability.

**Task statements:** TS 4.1 (explicit criteria), 4.2 (few-shot), 4.3 (structured output via tool use), 4.4 (validation and retry), 4.5 (Batches API), 5.5 (human review and calibration).

**Top 3 traps**
1. Believing a valid schema means correct values. A schema guarantees shape, not truth.
2. Retrying when the information is simply not in the document. The model will invent it.
3. Cutting human review because overall accuracy looks high, without checking by document type and field.

## 2. Architecture Diagram

```mermaid
flowchart TD
    Doc["Document"] --> Req["Request:<br>forced tool_choice + JSON schema<br>(claude-haiku-4-5, no thinking)"]
    Req --> Out["tool_use result"]
    Out --> Val{"Schema + semantic checks<br>(totals add up, enums, nulls)"}
    Val -->|"Valid"| Conf{"Calibrated confidence"}
    Val -->|"Fixable error"| Retry["Retry with document,<br>failed output, and error"]
    Retry --> Out
    Val -->|"Info missing in source"| Human["Human review<br>(no retry)"]
    Conf -->|"High"| Auto["Auto-commit + stratified sample"]
    Conf -->|"Medium"| Spot["Spot check"]
    Conf -->|"Low"| Human
    Bulk["Large backlog"] --> Batch["Batches API with custom_id"]
```

## 3. How a Good Answer Thinks

1. **Need guaranteed shape?** Use tool use with a JSON schema (`tool_choice` forced to the extraction tool). That removes syntax errors.
2. **Schema is not truth.** Add semantic checks: totals, dates, cross-field rules.
3. **Field may be absent?** Make it nullable or optional. Required fields tempt the model to invent values.
4. **Unknown categories?** Use an enum with `other` plus a detail string, and `unclear` where the document is ambiguous.
5. **Retry only when fixable.** Send the document, the failed extraction, and the specific error. If the data is not in the document, do not retry.
6. **Backlog, no rush?** Message Batches API (50% cheaper, up to 24h, `custom_id` to match results).
7. **Human review.** Route by calibrated confidence. Keep stratified sampling by document type and field, even for high confidence.
8. **Forced tool and thinking do not mix.** Forced `tool_choice` returns HTTP 400 with extended thinking, so the extraction call uses no thinking.

### Model and Sampling Notes
- Use `claude-sonnet-4-5` for coordinators and reasoning, `claude-haiku-4-5` for workers and extraction.
- With extended thinking on: omit `temperature`, leave `top_k` and `top_p` unset, and use only `tool_choice` `auto` or `none` (forced or `any` returns HTTP 400). Forced-tool extraction calls must not use thinking.
- In tool loops, return thinking blocks with their signature unchanged.
- Loops end on `stop_reason` only (`end_turn`, `tool_use`, `max_tokens`, `stop_sequence`, `pause_turn`, `refusal`). A max-iterations counter is a safety cap, not the stop rule.

## 4. Task Statement Map

| Task statement | How it shows up here | Trap |
|---|---|---|
| TS 4.1 Criteria | Explicit field rules and edge cases | Vague "extract carefully" |
| TS 4.2 Few-shot | Examples for odd layouts and ambiguous fields | No examples for edge cases |
| TS 4.3 Tool use + schema | Forced tool, nullable optional fields, `other` enum | Believing schema fixes wrong values |
| TS 4.4 Validate and retry | Retry with error feedback; stop when data is absent | Blind retries that cause hallucination |
| TS 4.5 Batches | Backlog with `custom_id` | Batching an interactive flow |
| TS 5.5 Human review | Confidence routing and stratified audits | Trusting one overall accuracy number |

## 5. Worked Example

**Document:** A drone inspection invoice. No tax ID. Subtotal `500`, tax `0`, total `500`.

1. The request forces `extract_invoice`. The schema marks `tax_id` as nullable.
2. The model returns `tax_id: null`. Schema check passes.
3. Semantic check: subtotal plus tax equals total. Passes.
4. A second invoice has total `120` but subtotal `100` and tax `10`. The check fails. This is fixable: retry with the document, the failed output, and the message "subtotal + tax != total".
5. A third invoice has no tax number anywhere, but the schema (wrongly) requires it. A retry would invent one. Instead, make the field nullable and send low-confidence items to a human.
6. Confidence `0.95` goes to auto-commit, with a sample for audit. `0.8` gets a spot check. `0.5` goes to a human.

Try it offline: `scenarios/scenario_6_structured_data_extraction/simulation.py` (`build_extraction_request`, `validate_against_schema`, `is_retry_eligible`, `route_by_confidence`).

## 7. Self-Check Checklist

- [ ] I can say what a schema guarantees and what it does not.
- [ ] I make absent-able fields nullable.
- [ ] I know when a retry helps and when it hurts.
- [ ] I can explain why forced tool choice and extended thinking do not combine.
- [ ] I keep stratified sampling even for high-confidence output.

**Memory hook:** "Schema for shape, checks for truth, null for missing, humans for the unsure."


In [ ]:
# Step 1: Environment Setup & Simulation Import
import sys
from pathlib import Path

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scenarios.scenario_6_structured_data_extraction.simulation import *

print(' Scenario Workbench Initialized: Scenario 6: Structured Data Extraction Pipeline')
print(' Zero live API calls, 100% offline determinism')


### 🧪 Step 2: Core Execution Flow & Happy Path Simulation
Demonstrating end-to-end pipeline execution with audit logging.


In [ ]:
# Test Math Error Recovery via Retry
pipeline = StructuredExtractionPipeline(random_seed=42)
doc_text = "Subtotal: $50, Tax: $5, Total: $55, Tax ID: TX-1"
turns = [
    {"subtotal": 50.0, "tax": 5.0, "total": 50.0, "tax_id": "TX-1"},
    {"subtotal": 50.0, "tax": 5.0, "total": 55.0, "tax_id": "TX-1"}
]
outcome = pipeline.process_extraction("d1", doc_text, turns, confidence_score=0.96)
print("Extraction Status:", outcome.status)
print("Retries Used:", outcome.retries_used)
print("Routing Tier:", outcome.routing_tier)
assert outcome.status == "SUCCESS"
assert outcome.retries_used == 1



### ⚠️ Step 3: Edge Case Testing & Guardrail Verifications
Testing boundary conditions, safety gates, error recovery, and escalation protocols.


In [ ]:
# Test Unrecoverable Missing Fact Aborts Retry Loop Immediately
pipeline = StructuredExtractionPipeline(random_seed=42)
doc_missing = "Simple receipt: Total $10. Paid in cash at diner counter."
turns_missing = [{"subtotal": 10.0, "tax": 0.0, "total": 10.0, "tax_id": None}]
outcome_missing = pipeline.process_extraction("d2", doc_missing, turns_missing, confidence_score=0.95)
print("Missing Fact Status:", outcome_missing.status)
print("Retries Used:", outcome_missing.retries_used)
assert outcome_missing.status == "FAILED_UNRECOVERABLE"
assert outcome_missing.retries_used == 0
assert outcome_missing.routing_tier == ExtractionRoutingTier.HUMAN_TRIAGE



### ✅ Step 4: Verification Summary & SLA Compliance
All assertions passed with zero unauthorized actions and 100% deterministic compliance.


### Step 5: New behaviour - forced-tool request, nullable schema, routing
Offline checks for the extended simulation.


In [ ]:
req = build_extraction_request("Invoice text")
assert req["tool_choice"]["type"] == "tool" and "thinking" not in req
ok = {"vendor": "A", "total": 10.0, "tax_id": None, "expense_category": "unclear"}
assert validate_against_schema(ok) == []
assert is_retry_eligible(False, True, 0) and not is_retry_eligible(False, False, 0)
pipe = StructuredExtractionPipeline(random_seed=1)
print("Routing 0.5:", pipe.route_by_confidence(0.5)[0].value)
